# Customer Churn Prediction - Exploratory Data Analysis
In this project, I aim to predict which telecom customers are likely to cancel their subscription(churn). By identifying at-risks customers, the business can intervene with retention offers, saving revenue.

In [ ]:
import numpy as  np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style = "whitegrid")

# Load the data
df = pd.read_csv("../data/telco_customer_churn.csv")

# Inspect data
display(df.head())
df.info()

In [ ]:
# Check for the missing values
print(df.isna().sum())

# Notice 'TotalCharges' might be an object type instead of float. Check it:
print(df["TotalCharges"].dtype)
# If it's an object, it might contain blank spaces instead of NaNs.
# Let's see unique values that aren't numbers.
print(df["TotalCharges"].unique()[:10])

In [ ]:
# Replace spaces(" ") into np.nan and convert column to float.
df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)
df["TotalCharges"] = df["TotalCharges"].astype(float)

print(df["TotalCharges"].dtype)
print(df["TotalCharges"].isna().sum())

In [ ]:
# Bar plot for the distribution of churn
fig,ax = plt.subplots(figsize = (6,4))
sns.countplot(data = df, x = "Churn", hue = "Churn", palette = "Set1", ax = ax )
ax.set_title("Distribution of Churn")
ax.legend(title = "Churn", labels = ["No","Yes"])

In [ ]:
# Percentage of customer churned vs stayed
print(df["Churn"].value_counts(normalize=True)*100)

The data is imbalance. Approximately 73% of the customers didnot churn, while 26% churned. Since the non churn class is much greater than churn class, the data is not evenly distributed.

## Data preprocessing
Machine learning model requires numerical data. I will drop unnecessary columns, handle the missing values, encode the target variable, and one-hot encode the categorial features. 

In [ ]:
# 1. Drop the customerID because (it's unique for every person and has no predictive power.)
df = df.drop(columns = ['customerID'])

# 2. Drop the 11 rows with missing TotalCharges (it's less than 0.2% of the data, so safe to drop)
df = df.dropna(subset = ['TotalCharges'])

# 3. Convert the target variable Churn to 1(Yes) and 0(No)
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

# 4. Convert categorical variables into dummy/indicator variables (One-Hot Encoding)
# We drop the first column to avoid the "dummy variable trap" (multicollinearity)
df_encoded = pd.get_dummies(df, drop_first=True)

print(f"Original Shape: {df.shape}")
print(f"Encoded Shape: {df_encoded.shape}")
display(df_encoded.head())

Now that the data is fully numerical, I will separate it into Features(X) and Target(y) and split it into Training and Testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Separate Features and Target
X = df_encoded.drop(columns = ['Churn'])
y = df_encoded["Churn"]

# Train/Test split(80% train and 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

print(f"Training Set: {X_train.shape}")
print(f"Testing Set: {X_test.shape}")

The number of columns increased because One-Hot Encoding converted categorical columns into multiple binary (0/1) columns. For example, a single column such as InternetService was split into separate columns like InternetService_Fiber optic and InternetService_No, allowing machine learning models to work with categorical data in numeric form.

## Baseline Modeling: Logistic Regression
I will train a Logistic Regression model, because dataset is imbalanced(73% No, 27% Yes), I will use "class_weight='balanced'" so the model doesn't just ignore the minority class. I will evaluate the model using Precision, Recall, and the Confusion Matrix.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Initialize the model with class_weight='balanced'
log_reg = LogisticRegression(max_iter=1000, class_weight = 'balanced', random_state=42)

# 2. Train the model
log_reg.fit(X_train,y_train)

# 3. Make the prediction
y_pred = log_reg.predict(X_test)

# 4. Evaluate
print(f"Accuracy: {accuracy_score(y_test,y_pred)*100:.2f}%")
print("Classification Report:")
print(classification_report(y_test,y_pred))

print("\n Confusion matrix:")
print(confusion_matrix(y_test,y_pred))



## Feature Scaling
This model reached the iteration limit because the features like TotalCharges (thousands) and SeniorCitizen(0/1) are on vastly different scales. I will use StandardScale to normalize the training data, which helps the model to converge faster and find a better solution.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Initialize the scaler
scaler = StandardScaler()

# 2. Fit on the training data and transform both train and test sets
# We ONLY fit on X_train to avoid the data leakage from test set!
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Re-train the Logistic Regression on SCALED data.
# (We can also drop max iter since it will converge easily, but we'll leave it for safety.)
log_reg_scaled = LogisticRegression(class_weight='balanced', random_state=42)
log_reg_scaled.fit(X_train_scaled, y_train)

# 4. Make new predictions
y_pred_scaled = log_reg_scaled.predict(X_test_scaled)

# 5. Evalute
print(f"Scaled Accuracy score: {accuracy_score(y_test, y_pred_scaled)*100:.2f}%")

print("Scaled Classification Report:")
print(classification_report(y_test, y_pred_scaled))

print("Scaled Confusion matrix: \n")
print(confusion_matrix(y_test, y_pred_scaled))

After applying StandardScaler, Logistic Regression converged successfully without warnings. Scaling mainly improved the training process and convergence speed, while the model accuracy remained approximately the same.